# 🏆 Previsão da 1ª Rodada — Copa do Mundo FIFA 2026
## Pipeline com MLP (TensorFlow/Keras)

**Arquitetura:** `Input(12) → Dense(128) → Dense(64) → Dense(32) → Softmax(25)`  
**Dados de treino:** 7.515 jogos históricos entre seleções da Copa (1872–2026)  
**Dados de features:** Ranking FIFA, Elo Rating, Valor de Mercado, Odds  
**Rótulo:** Placar (0–4 × 0–4) como classificação multiclasse (25 classes)

---

## Sumário
1. Configuração e sementes  
2. Dados das seleções (features estáticas)  
3. Histórico de jogos (treino/teste)  
4. Engenharia de features por confronto  
5. Rótulos — placar como classificação multiclasse  
6. Split temporal: treino (até 2021) / teste (2022–2026)  
7. Arquitetura MLP  
8. Treinamento e curvas de aprendizado  
9. Avaliação no conjunto de teste  
10. Modelo final e previsão Copa 2026  
11. Exportação JSON  


Professor as minhas habilidades pythonisticas ainda não estão muito bem desenvolvidas, por este motivo o claude me deu uma força na criação do código.
Foram utilizados os seguintes dados: Ranking da fifa (https://inside.fifa.com/fifa-world-ranking/men), ranking elo (https://eloratings.net/), Valor de mercado das seleções (https://www.transfermarkt.com.br/vereins-statistik/wertvollstenationalmannschaften/marktwertetop) e as odd apenas para a primeira rodada (https://www.oddschecker.com/football/world-cup). 
Além deles foi utilizado o dataset do Kaggle com todos os resultados de jogos de 1872 até 2026 (https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017). Busquei dados importantes das previsões do Polvo Paul (https://pt.wikipedia.org/wiki/Paul_(polvo)), visto que alcançou 100% de acerto nos palpites para a copa da África do Sul, porém o molusco teve uma vida breve e poucos prognósticos em sua carreira.

Todos os dados foram coletados no dia (04/06/2026).
O tratamento dos dados foi realizado no claude.ai. Primeiramente, foram concatenadas as 3 bases de dados: Ranking FIFA, Transfermarket e Ranking Elo. As colunas utilizadas foram: Sigla (Mapeamento fornecido), País (Nome em português - chave comum), Rank FIFA (posição fifa_ranking), Pts FIFA (pontos fifa_ranking), Valor Mercado (Transfermarkt), Elo Rating (elo_ratings), Gols Pró (Gols Marcados) e Gols Contra (Gols Sofridos).

Em seguida foram tratados os dados de bets (as odds): Tratando como exemplo as odds de MEX x RSA (ML: Moneyline):
ML Vitória México : -260 
ML Empate : 360 
ML Vitória RSA : 750
 
1. Probabilidade Implícita Bruta: A odd em moneyline americano carrega embutida a probabilidade que a casa de apostas atribui àquele resultado. A fórmula varia conforme o sinal:
	Odds negativas (favorito): P= ∣ML∣ / ∣ML∣ +100​ 
Odds positivas (azarão): P= 100 / ML + 100​ 
Aplicando para MEX × RSA:
México  → -260 → P = 260 / (260+100) = 260/360 = 72,2%
Empate  →  360 → P = 100 / (360+100) = 100/460 = 21,7%
RSA     →  750 → P = 100 / (750+100) = 100/850 = 11,8%
2. Overround (margem da casa): Se as probabilidades fossem "justas", a soma das três deveria ser exatamente 100%. Mas a casa de apostas inflaciona todas elas levemente para garantir lucro independente do resultado. Esse excesso é o overround: 72,2% + 21,7% + 11,8% = 105,7% Overround = 105,7% - 100% = 5,7% 
3. Probabilidade Normalizada (limpa): Para usar as odds como input no modelo de ML sem esse viés, precisamos remover o overround. Fazemos isso dividindo cada probabilidade bruta pela soma total:
Pnorm​= Pbruta / P1​+Pe​+P2​​​ 
Total = 105,7% = 1,057 
México → 72,2% / 1,057 = 68,3% 
Empate → 21,7% / 1,057 = 20,6% 
RSA → 11,8% / 1,057 = 11,2% 
Soma → 68,3% + 20,6% + 11,2% = 100,0%

Em seguida tratamos o arquivo dataset do Kaggle com os resultados de jogos de seleções desde 1872 até 2026 (https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017) para results_copa2026.csv, deixando apenas com as 48 seleções da copa. O resultado foi de 49.378 jogos para 7.587 jogo.

Os datasets completos e sua utilização foram adicionados na seçao 4.1.



## 1. Configuração do Ambiente e Sementes

In [2]:
import os, json, random, warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ── Sementes aleatórias (reprodutibilidade) ──
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Dados das Seleções (Features Estáticas)

### Fontes
| Feature | Fonte | Justificativa |
|---|---|---|
| `rank_fifa` | FIFA World Ranking (abr/2026) | Métrica oficial de força |
| `pontos_fifa` | Mesma fonte | Mais granular que a posição |
| `valor_mercado_meur` | Transfermarkt (jun/2026) | Proxy de qualidade do elenco (€ mi) |
| `elo_rating` | eloratings.net (jun/2026) | Rating dinâmico, melhor preditor histórico |
| `gols_pro_hist` | eloratings.net | Volume ofensivo acumulado |
| `gols_contra_hist` | eloratings.net | Volume defensivo acumulado |
| `prob_norm_*` | Odds (jun/2026) | Consenso de mercado, normalizado sem overround |


In [3]:
# ── Carregar dataset de seleções diretamente do XLSX ──
try:
    df_raw = pd.read_excel('selecoes_copa2026_completo.xlsx', header=1)
    print("✅ selecoes_copa2026_completo.xlsx carregado (pasta local)")
except FileNotFoundError:
    df_raw = pd.read_excel('/mnt/user-data/outputs/selecoes_copa2026_completo.xlsx', header=1)
    print("✅ selecoes_copa2026_completo.xlsx carregado (path absoluto)")

# ── Selecionar e renomear apenas as colunas relevantes para o MLP ──
# Excluídas: País, Adversário, Jogo, ML Própria/Empate/Adversária,
#            Odd Própria/Empate/Adversária, Overround %, Favorito
df_sel = df_raw[[
    'Sigla',
    'Rank FIFA',
    'Pts FIFA',
    'Valor Mercado',
    'Elo Rating',
    'Gols Pró',
    'Gols Contra',
    'P.Norm Própria',
    'P.Norm Empate',
    'P.Norm Adversário',
]].copy()

df_sel.columns = [
    'sigla',
    'rank_fifa',
    'pontos_fifa',
    'valor_mercado_str',   # ainda é string — será convertido abaixo
    'elo_rating',
    'gols_pro_hist',
    'gols_contra_hist',
    'prob_norm_propria',
    'prob_norm_empate',
    'prob_norm_adversaria',
]

# ── Tratar tipos ──────────────────────────────────────────────────────────────

# Valor Mercado: "€ 1.55 Bi." → float em milhões de €
def parse_mercado(val):
    import re
    val = str(val).replace('€','').replace(' ','').replace(',','.')
    if 'Bi.' in val:
        return float(val.replace('Bi.','')) * 1000
    elif 'mi.' in val:
        return float(val.replace('mi.',''))
    return float('nan')

df_sel['valor_mercado_meur'] = df_sel['valor_mercado_str'].apply(parse_mercado)
df_sel = df_sel.drop(columns=['valor_mercado_str'])

# Probabilidades: "65.9%" → 0.659
for col in ['prob_norm_propria', 'prob_norm_empate', 'prob_norm_adversaria']:
    df_sel[col] = df_sel[col].astype(str).str.replace('%','').astype(float) / 100

# Demais colunas numéricas
for col in ['rank_fifa','pontos_fifa','elo_rating','gols_pro_hist','gols_contra_hist']:
    df_sel[col] = pd.to_numeric(df_sel[col])

# Índice = sigla (facilita lookup por df_sel.loc['BRA'])
df_sel = df_sel.set_index('sigla')

print(f"Shape    : {df_sel.shape}")
print(f"NaNs     : {df_sel.isnull().sum().sum()}")
print(f"\nColunas usadas no modelo:")
for c in df_sel.columns:
    print(f"  {c:<25} dtype={df_sel[c].dtype}")
print(f"\nTop 5 por Elo Rating:")
print(df_sel.sort_values('elo_rating', ascending=False).head(5)[
    ['rank_fifa','pontos_fifa','elo_rating','valor_mercado_meur']
].round(2))


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/selecoes_copa2026_completo.xlsx'

## 3. Histórico de Jogos (1872–2026)

**Fonte:** Dataset `results.csv` filtrado para confrontos entre as 48 seleções da Copa 2026.  
São **7.515 jogos** com placar válido, cobrindo mais de 150 anos de futebol internacional.


In [ ]:
# ── Carregar histórico de resultados ──
try:
    df_hist = pd.read_csv('results_copa2026.csv')
    print(f"✅ results_copa2026.csv carregado: {len(df_hist)} jogos")
except FileNotFoundError:
    # Fallback: carregar do caminho original
    try:
        df_hist = pd.read_csv('/mnt/user-data/outputs/results_copa2026.csv')
        print(f"✅ results_copa2026.csv carregado (path absoluto): {len(df_hist)} jogos")
    except:
        print("❌ Arquivo não encontrado. Ajuste o caminho do CSV.")
        df_hist = pd.DataFrame()

if len(df_hist) > 0:
    df_hist['date'] = pd.to_datetime(df_hist['date'])
    df_hist['year'] = df_hist['date'].dt.year
    df_hist['home_score'] = df_hist['home_score'].astype(int)
    df_hist['away_score'] = df_hist['away_score'].astype(int)

    print(f"Período  : {df_hist['date'].min().date()} → {df_hist['date'].max().date()}")
    print(f"NaNs     : {df_hist.isnull().sum().sum()}")
    print(f"Amostra  :")
    print(df_hist.sample(5, random_state=SEED)[
        ['date','home_sigla','home_team','away_sigla','away_team','home_score','away_score']
    ].to_string(index=False))

    # Distribuição de placares
    print(f"\nDistribuição de gols por jogo:")
    print(f"  Média gols casa  : {df_hist['home_score'].mean():.2f}")
    print(f"  Média gols fora  : {df_hist['away_score'].mean():.2f}")
    print(f"  % vitória casa   : {(df_hist['home_score'] > df_hist['away_score']).mean():.1%}")
    print(f"  % empate         : {(df_hist['home_score'] == df_hist['away_score']).mean():.1%}")
    print(f"  % vitória fora   : {(df_hist['home_score'] < df_hist['away_score']).mean():.1%}")


## 4. Engenharia de Features por Confronto

Para cada jogo (casa × fora), derivamos **diferenças e razões** entre os indicadores das duas seleções.  
Diferenças capturam *quem é mais forte*, e não apenas valores absolutos.

| Feature | Fórmula | Interpretação |
|---|---|---|
| `delta_elo` | ELO_casa − ELO_fora | + → casa favorita pelo Elo |
| `delta_pontos_fifa` | pts_casa − pts_fora | + → casa favorita pelo FIFA |
| `delta_rank_fifa` | rank_fora − rank_casa | + → casa melhor posicionada |
| `log_ratio_mercado` | log(val_casa / val_fora) | + → casa com elenco mais valioso |
| `delta_gols_pro` | gols_pro_casa − gols_pro_fora | + → casa mais ofensiva |
| `delta_gols_contra` | gols_contra_fora − gols_contra_casa | + → casa mais sólida defensivamente |
| `prob_vitoria_casa` | P(casa vence) normalizada | probabilidade implícita de mercado |
| `prob_empate` | P(empate) normalizada | probabilidade implícita de mercado |
| `prob_vitoria_fora` | P(fora vence) normalizada | probabilidade implícita de mercado |

> **Tratamento de odds no histórico:** jogos históricos (1872–2026) não têm odds disponíveis.  
> Para esses jogos, as 3 features de probabilidade recebem o valor neutro **1/3**,  
> indicando ausência de informação de mercado. Nos 24 jogos de 2026, as odds reais são usadas.


In [ ]:
NEUTRO = (1/3, 1/3, 1/3)

# Mapa sigla → odds normalizadas (apenas para os 24 jogos de 2026)
ODDS_2026 = {
    ('MEX','RSA'): (0.683, 0.206, 0.111), ('KOR','CZE'): (0.363, 0.295, 0.343),
    ('CAN','BIH'): (0.531, 0.266, 0.203), ('USA','PAR'): (0.488, 0.280, 0.232),
    ('QAT','SUI'): (0.073, 0.147, 0.780), ('BRA','MAR'): (0.590, 0.249, 0.161),
    ('HAI','SCO'): (0.146, 0.211, 0.643), ('AUS','TUR'): (0.183, 0.272, 0.546),
    ('GER','CUW'): (0.940, 0.048, 0.013), ('NED','JPN'): (0.477, 0.265, 0.258),
    ('CIV','ECU'): (0.262, 0.337, 0.401), ('SWE','TUN'): (0.494, 0.286, 0.219),
    ('ESP','CPV'): (0.874, 0.092, 0.034), ('BEL','EGY'): (0.581, 0.245, 0.174),
    ('KSA','URU'): (0.135, 0.215, 0.650), ('IRN','NZL'): (0.499, 0.280, 0.221),
    ('FRA','SEN'): (0.659, 0.206, 0.135), ('IRQ','NOR'): (0.072, 0.145, 0.783),
    ('ARG','ALG'): (0.693, 0.207, 0.100), ('AUT','JOR'): (0.729, 0.171, 0.101),
    ('POR','COD'): (0.760, 0.161, 0.079), ('ENG','CRO'): (0.559, 0.255, 0.185),
    ('GHA','PAN'): (0.464, 0.280, 0.257), ('UZB','COL'): (0.112, 0.202, 0.686),
}

def get_odds(casa, fora):
    return ODDS_2026.get((casa, fora),
           ODDS_2026.get((fora, casa), NEUTRO))

def features_do_jogo(casa, fora, usar_odds=False):
    """
    Retorna vetor de 9 features para um par (time_casa, time_fora).
    usar_odds=True → usa odds reais (apenas para predição 2026).
    usar_odds=False → odds neutras 1/3 (para dados históricos sem mercado).
    """
    c = df_sel.loc[casa]
    f = df_sel.loc[fora]

    if usar_odds:
        pv, pe, pf = get_odds(casa, fora)
    else:
        pv, pe, pf = NEUTRO

    return np.array([
        c['elo_rating']          - f['elo_rating'],           # delta_elo
        c['pontos_fifa']         - f['pontos_fifa'],           # delta_pontos_fifa
        f['rank_fifa']           - c['rank_fifa'],             # delta_rank_fifa
        np.log(max(c['valor_mercado_meur'],1) /
               max(f['valor_mercado_meur'],1)),                # log_ratio_mercado
        c['gols_pro_hist']       - f['gols_pro_hist'],         # delta_gols_pro
        f['gols_contra_hist']    - c['gols_contra_hist'],      # delta_gols_contra
        pv, pe, pf                                             # probs de mercado
    ], dtype=np.float32)

FEATURE_NAMES = [
    'delta_elo', 'delta_pontos_fifa', 'delta_rank_fifa',
    'log_ratio_mercado', 'delta_gols_pro', 'delta_gols_contra',
    'prob_vitoria_casa', 'prob_empate', 'prob_vitoria_fora',
]
N_FEATURES = len(FEATURE_NAMES)

# Exemplo: BRA × MAR
print(f"Features ({N_FEATURES} total) — exemplo BRA × MAR:")
ex = features_do_jogo('BRA', 'MAR', usar_odds=True)
for n, v in zip(FEATURE_NAMES, ex):
    print(f"  {n:25s}: {v:+.4f}")


## 4.1. Dicionário de Dados — Tabulação das Planilhas de Origem

Antes de prosseguir, esta seção documenta **todas as colunas** dos dois datasets de origem e como cada uma é utilizada (ou não) no pipeline. Nenhuma coluna entra "crua" no MLP — todas as features são diferenças/razões calculadas entre os dois times do confronto (ver Seção 4).

### Legenda — coluna "Status"

| Status | Significado |
|---|---|
| **Identificador** | Coluna de identificação; não entra no modelo |
| **Base p/ cálculo** | Valor usado para calcular uma feature derivada (diferença ou razão entre os dois times) |
| **Feature MLP** | Feature derivada que entra diretamente no vetor de entrada do MLP |
| **Não usada** | Coluna informativa; não entra no modelo de nenhuma forma |

---

### Dataset 1 — `selecoes_copa2026_completo.xlsx` (48 linhas × 21 colunas)

| # | Coluna | Tipo | Exemplo | Descrição | Status | Feature Derivada |
|---|---|---|---|---|---|---|
| 1 | `Sigla` | str | FRA | Código FIFA de 3 letras — chave primária para lookup das features | Identificador | — |
| 2 | `País` | str | França | Nome da seleção em português | Não usada | — |
| 3 | `Adversário` | str | SEN | Sigla do adversário na 1ª rodada | Não usada | — |
| 4 | `Jogo` | str | FRA × SEN | Confronto da 1ª rodada no formato Sigla1 × Sigla2 | Não usada | — |
| 5 | `Rank FIFA` | int | 1 | Posição no Ranking FIFA (abr/2026) | Base p/ cálculo | `delta_rank_fifa = Rank_fora − Rank_casa` |
| 6 | `Pts FIFA` | float | 1877.32 | Pontuação numérica no Ranking FIFA | Base p/ cálculo | `delta_pontos_fifa = Pts_casa − Pts_fora` |
| 7 | `Valor Mercado` | str | € 1.55 Bi. | Valor de mercado do elenco. Texto → conversão para float (€ mi) | Base p/ cálculo | `log_ratio_mercado = log(Mercado_casa / Mercado_fora)` |
| 8 | `Elo Rating` | int | 2062 | Rating Elo atual (eloratings.net, jun/2026). Escala ~1270–2165 | Base p/ cálculo | `delta_elo = ELO_casa − ELO_fora` |
| 9 | `Gols Pró` | int | 1707 | Total de gols marcados no histórico completo da seleção | Base p/ cálculo | `delta_gols_pro = GolsPró_casa − GolsPró_fora` |
| 10 | `Gols Contra` | int | 1274 | Total de gols sofridos no histórico completo da seleção | Base p/ cálculo | `delta_gols_contra = GolsContra_fora − GolsContra_casa` |
| 11 | `ML Própria` | int | -230 | Moneyline americano para vitória da seleção. Sintetizado pela P.Norm | Não usada | — |
| 12 | `ML Empate` | int | 360 | Moneyline americano para empate. Sintetizado pela P.Norm | Não usada | — |
| 13 | `ML Adversário` | int | 600 | Moneyline americano para vitória do adversário | Não usada | — |
| 14 | `P.Norm Própria` | str | 65.9% | Prob. de vitória da seleção, normalizada (overround removido) | Base p/ cálculo | `prob_vitoria_casa` ou `prob_vitoria_fora` (conforme posição no jogo) |
| 15 | `P.Norm Empate` | str | 20.6% | Prob. de empate, normalizada | Base p/ cálculo | `prob_empate` (igual para os dois times do confronto) |
| 16 | `P.Norm Adversário` | str | 13.5% | Prob. de vitória do adversário, normalizada | Base p/ cálculo | `prob_vitoria_fora` ou `prob_vitoria_casa` (invertida) |
| 17 | `Odd Própria` | float | 1.52 | Odd decimal = 1 / P.Norm Própria. Informativo | Não usada | — |
| 18 | `Odd Empate` | float | 4.86 | Odd decimal = 1 / P.Norm Empate. Informativo | Não usada | — |
| 19 | `Odd Adversário` | float | 7.40 | Odd decimal = 1 / P.Norm Adversário. Informativo | Não usada | — |
| 20 | `Overround %` | str | 5.72% | Margem da casa de apostas. Já removida nas P.Norm. Informativo | Não usada | — |
| 21 | `Favorito` | str | França | Nome da seleção com maior P.Norm. Informativo | Não usada | — |

**Observações de tratamento:**
- **Valor Mercado**: string `"€ 1.55 Bi."` → `1550.0` (float, milhões de €). `"Bi."` × 1.000, `"mi."` × 1.
- **P.Norm \***: string `"65.9%"` → `0.659` (float / 100).
- **ML \* / Odd \* / Overround % / Favorito**: ignoradas pelo pipeline.

---

### Dataset 2 — `results_copa2026.csv` (7.515 linhas × 7 colunas)

| # | Coluna | Tipo | Exemplo | Descrição | Status | Feature Derivada |
|---|---|---|---|---|---|---|
| 1 | `date` | str | 1872-11-30 | Data do jogo (AAAA-MM-DD). Usada para o split temporal | Base p/ cálculo | Ano → split treino (≤2021) / teste (≥2022) |
| 2 | `home_sigla` | str | SCO | Sigla do time mandante — chave para lookup das features no `df_sel` | Base p/ cálculo | Identifica time_casa → todas as 9 features derivadas |
| 3 | `home_team` | str | Scotland | Nome do time mandante em inglês. Apenas para auditoria | Não usada | — |
| 4 | `away_sigla` | str | ENG | Sigla do time visitante — chave para lookup das features no `df_sel` | Base p/ cálculo | Identifica time_fora → todas as 9 features derivadas |
| 5 | `away_team` | str | England | Nome do time visitante em inglês. Apenas para auditoria | Não usada | — |
| 6 | `home_score` | int | 2 | Gols do mandante (tempo regulamentar) | Base p/ cálculo | Rótulo = `min(home_score,4)-min(away_score,4)` → 25 classes |
| 7 | `away_score` | int | 1 | Gols do visitante (tempo regulamentar) | Base p/ cálculo | Rótulo = `min(home_score,4)-min(away_score,4)` → 25 classes |

---

### Resumo — Vetor de Entrada do MLP (9 features)

| # | Feature (MLP) | Fórmula | Colunas base |
|---|---|---|---|
| 1 | `delta_elo` | `ELO_casa − ELO_fora` | Elo Rating (selecoes) |
| 2 | `delta_pontos_fifa` | `Pts_casa − Pts_fora` | Pts FIFA (selecoes) |
| 3 | `delta_rank_fifa` | `Rank_fora − Rank_casa` | Rank FIFA (selecoes) |
| 4 | `log_ratio_mercado` | `log(ValMercado_casa / ValMercado_fora)` | Valor Mercado (selecoes) |
| 5 | `delta_gols_pro` | `GolsPro_casa − GolsPro_fora` | Gols Pró (selecoes) |
| 6 | `delta_gols_contra` | `GolsContra_fora − GolsContra_casa` | Gols Contra (selecoes) |
| 7 | `prob_vitoria_casa` | P.Norm Própria do time casa | P.Norm Própria/Adversário — 1/3 no histórico |
| 8 | `prob_empate` | P.Norm Empate | P.Norm Empate — 1/3 no histórico |
| 9 | `prob_vitoria_fora` | P.Norm Adversária do time casa | P.Norm Adversário/Própria — 1/3 no histórico |


## 5. Rótulos — Placar como Classificação Multiclasse

Cada placar (gols_casa, gols_fora) é tratado como uma **classe discreta**.  
Aplicamos **clipping a 4 gols** por time → grade (0–4) × (0–4) = **25 classes**.

Historicamente, >95% dos jogos internacionais terminam com ≤ 4 gols por time.


In [ ]:
MAX_GOLS = 4
TODAS_CLASSES = [f"{gc}-{gf}"
                 for gc in range(MAX_GOLS+1)
                 for gf in range(MAX_GOLS+1)]
N_CLASSES = len(TODAS_CLASSES)

label_encoder = LabelEncoder()
label_encoder.fit(TODAS_CLASSES)

def placar_para_label(gc, gf):
    return f"{min(gc, MAX_GOLS)}-{min(gf, MAX_GOLS)}"

def label_para_placar(label):
    gc, gf = label.split('-')
    return int(gc), int(gf)

def resultado(label):
    gc, gf = label_para_placar(label)
    return 'V' if gc > gf else ('D' if gc < gf else 'E')

print(f"Número de classes: {N_CLASSES}")
print(f"Classes: {TODAS_CLASSES}")

# ── Construir dataset com features + rótulos ──
X_list, y_list, anos_list, pares_list = [], [], [], []

for _, row in df_hist.iterrows():
    casa = row['home_sigla']
    fora = row['away_sigla']
    if casa not in df_sel.index or fora not in df_sel.index:
        continue
    feat  = features_do_jogo(casa, fora, usar_odds=False)
    label = placar_para_label(int(row['home_score']), int(row['away_score']))
    X_list.append(feat)
    y_list.append(label)
    anos_list.append(int(row['year']))
    pares_list.append((casa, fora))

X_all  = np.array(X_list, dtype=np.float32)
y_all  = label_encoder.transform(y_list)
anos   = np.array(anos_list)

print(f"\nAmostras válidas: {len(X_all)}")
print(f"\nTop 10 placares mais frequentes:")
pd.Series(y_list).value_counts().head(10).rename('jogos').to_frame().rename_axis('placar').reset_index().pipe(
    lambda d: print(d.to_string(index=False))
)


## 6. Split Temporal: Treino / Teste

**Estratégia:** split por ano — respeita a natureza temporal e evita vazamento.

- **Treino:** jogos até 2021 (inclusive)  
- **Teste:** jogos de 2022 em diante (inclui a Copa 2022 e amistosos recentes)


In [ ]:
mask_treino = anos <= 2021
mask_teste  = anos >= 2022

X_treino = X_all[mask_treino]; y_treino = y_all[mask_treino]
X_teste  = X_all[mask_teste];  y_teste  = y_all[mask_teste]

print(f"Treino : {len(X_treino):5d} jogos  (até 2021)")
print(f"Teste  : {len(X_teste):5d} jogos  (2022–2026)")
print(f"\nDistribuição resultado — Treino:")
rt = [resultado(l) for l in label_encoder.inverse_transform(y_treino)]
for r, n in pd.Series(rt).value_counts().items():
    print(f"  {r}: {n} ({n/len(rt):.1%})")

# ── Normalização (StandardScaler ajustado apenas no treino) ──
scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

print(f"\nMédia features (treino) : {scaler.mean_.round(2)}")
print(f"Desvio-padrão (treino)  : {scaler.scale_.round(2)}")


## 7. Arquitetura MLP

```
Input (9 features)
    │
Dense(128, relu) + BatchNormalization + Dropout(0.3)
    │
Dense(64,  relu) + BatchNormalization + Dropout(0.2)
    │
Dense(32,  relu)
    │
Dense(25,  softmax)  ← 25 placares possíveis (0-4 × 0-4)
```

**Escolhas de projeto:**
- **ReLU** — padrão para MLPs tabulares
- **BatchNormalization** — estabiliza gradientes com dataset grande
- **Dropout** — regularização contra overfitting
- **Softmax** — probabilidades somam 1 sobre as 25 classes
- **Sparse Categorical Cross-Entropy** — perda para labels inteiros


In [ ]:
def construir_mlp(n_features=N_FEATURES, n_classes=N_CLASSES, lr=1e-3):
    tf.random.set_seed(SEED)
    modelo = keras.Sequential([
        layers.Input(shape=(n_features,), name='entrada'),

        layers.Dense(128, activation='relu', name='densa_128'),
        layers.BatchNormalization(name='bn_128'),
        layers.Dropout(0.3, name='dropout_128', seed=SEED),

        layers.Dense(64, activation='relu', name='densa_64'),
        layers.BatchNormalization(name='bn_64'),
        layers.Dropout(0.2, name='dropout_64', seed=SEED),

        layers.Dense(32, activation='relu', name='densa_32'),

        layers.Dense(n_classes, activation='softmax', name='saida'),
    ], name='MLP_Copa_2026')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return modelo

modelo = construir_mlp()
modelo.summary()


## 8. Treinamento e Curvas de Aprendizado

In [ ]:
EPOCHS    = 300
BATCH     = 64
VAL_SPLIT = 0.15

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=40,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=20, min_lr=1e-5, verbose=0
    ),
]

tf.random.set_seed(SEED)
historico = modelo.fit(
    X_treino_sc, y_treino,
    epochs=EPOCHS,
    batch_size=BATCH,
    validation_split=VAL_SPLIT,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nEpoca final: {len(historico.history['loss'])}")
print(f"Melhor val_loss: {min(historico.history['val_loss']):.4f}")
print(f"Melhor val_acc : {max(historico.history['val_accuracy']):.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(historico.history['loss'],     label='Treino',    color='steelblue', lw=2)
ax.plot(historico.history['val_loss'], label='Validação', color='tomato',    lw=2)
ax.set_title('Loss (Cross-Entropy)', fontsize=13, fontweight='bold')
ax.set_xlabel('Época'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(historico.history['accuracy'],     label='Treino',    color='steelblue', lw=2)
ax.plot(historico.history['val_accuracy'], label='Validação', color='tomato',    lw=2)
ax.set_title('Acurácia', fontsize=13, fontweight='bold')
ax.set_xlabel('Época'); ax.set_ylabel('Acurácia')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Curvas de Aprendizado — MLP Copa 2026', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figura salva: curvas_aprendizado.png")


## 9. Avaliação no Conjunto de Teste (2022–2026)

Três métricas complementares:
- **Acurácia de placar exato** — placar previsto bate exatamente com o real
- **Acurácia de resultado (V/E/D)** — acerta se foi vitória, empate ou derrota
- **Top-3 accuracy** — o placar real estava entre os 3 mais prováveis


In [ ]:
y_pred_proba = modelo.predict(X_teste_sc, verbose=0)
y_pred       = y_pred_proba.argmax(axis=1)

acc_exato = accuracy_score(y_teste, y_pred)

res_real = [resultado(label_encoder.inverse_transform([y])[0]) for y in y_teste]
res_pred = [resultado(label_encoder.inverse_transform([y])[0]) for y in y_pred]
acc_res  = accuracy_score(res_real, res_pred)

top3 = sum(y in y_pred_proba[i].argsort()[-3:]
           for i, y in enumerate(y_teste))
acc_top3 = top3 / len(y_teste)

print("=" * 45)
print("  AVALIAÇÃO — CONJUNTO DE TESTE (2022–2026)")
print("=" * 45)
print(f"  Acurácia placar exato : {acc_exato:.2%}")
print(f"  Acurácia resultado    : {acc_res:.2%}")
print(f"  Top-3 accuracy        : {acc_top3:.2%}")
print(f"  Amostras de teste     : {len(y_teste)}")
print("=" * 45)

# ── Matriz de resultado ──
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(res_real, res_pred, labels=['V','E','D'])
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(['V','E','D']); ax.set_yticklabels(['V','E','D'])
ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão (V/E/D)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Modelo Final e Previsão Copa 2026

O modelo final é re-treinado em **todos os dados históricos** (treino + teste),  
maximizando o aprendizado antes de prever os 24 jogos da 1ª rodada.

Na predição final, as **odds reais** de 2026 são injetadas nas 3 features de probabilidade,  
substituindo o valor neutro 1/3 usado no treinamento histórico.


In [ ]:
# ── Re-treinar com todos os dados ──
scaler_final = StandardScaler()
X_all_sc = scaler_final.fit_transform(X_all)

modelo_final = construir_mlp(lr=5e-4)
tf.random.set_seed(SEED)
modelo_final.fit(
    X_all_sc, y_all,
    epochs=300,
    batch_size=64,
    validation_split=0.08,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=40, restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=20, min_lr=1e-5, verbose=0),
    ],
    verbose=0,
)
print("✅ Modelo final treinado em todos os dados históricos.")

# ── 24 Jogos da 1ª rodada Copa 2026 ──
JOGOS_2026 = [
    ('jogo1',  'MEX','RSA'), ('jogo2',  'KOR','CZE'), ('jogo3',  'CAN','BIH'),
    ('jogo4',  'USA','PAR'), ('jogo5',  'HAI','SCO'), ('jogo6',  'AUS','TUR'),
    ('jogo7',  'BRA','MAR'), ('jogo8',  'QAT','SUI'), ('jogo9',  'CIV','ECU'),
    ('jogo10', 'GER','CUW'), ('jogo11', 'NED','JPN'), ('jogo12', 'SWE','TUN'),
    ('jogo13', 'KSA','URU'), ('jogo14', 'ESP','CPV'), ('jogo15', 'IRN','NZL'),
    ('jogo16', 'BEL','EGY'), ('jogo17', 'FRA','SEN'), ('jogo18', 'IRQ','NOR'),
    ('jogo19', 'ARG','ALG'), ('jogo20', 'AUT','JOR'), ('jogo21', 'GHA','PAN'),
    ('jogo22', 'ENG','CRO'), ('jogo23', 'POR','COD'), ('jogo24', 'UZB','COL'),
]

resultados_2026 = {}
linhas = []

print(f"\n{'Jogo':<14} {'Placar':>6}  {'Top-3 (prob. modelo)':<45}  {'Odds mercado'}")
print("-"*95)

for jid, casa, fora in JOGOS_2026:
    # Features COM odds reais de 2026
    feat    = features_do_jogo(casa, fora, usar_odds=True).reshape(1, -1)
    feat_sc = scaler_final.transform(feat)
    proba   = modelo_final.predict(feat_sc, verbose=0)[0]

    top3_idx    = proba.argsort()[-3:][::-1]
    top3_labels = label_encoder.inverse_transform(top3_idx)
    top3_probs  = proba[top3_idx]

    placar = top3_labels[0]
    gc, gf = label_para_placar(placar)

    resultados_2026[jid] = {
        'casa': casa, 'fora': fora,
        'placar_previsto': placar,
        'gols_casa': gc, 'gols_fora': gf,
        'prob_modelo': round(float(top3_probs[0]), 4),
        'top3': [f"{l}({p:.1%})" for l, p in zip(top3_labels, top3_probs)],
    }

    pv, pe, pf = get_odds(casa, fora)
    t3 = ' | '.join([f"{l}({p:.1%})" for l, p in zip(top3_labels, top3_probs)])
    print(f"  {casa}×{fora:<4}  → {placar:>4}   {t3:<45}  V:{pv:.0%} E:{pe:.0%} F:{pf:.0%}")


## 11. Exportação do JSON Final

In [ ]:
saida_json = {
    "nome":  "Rafael Monteiro da Cruz",
    "turma": "Deep Learning E Processamento de Linguagem Natural - 2º BIM 2026",
    "resultados": {}
}

for jid, casa, fora in JOGOS_2026:
    pred = resultados_2026[jid]
    saida_json["resultados"][jid] = {
        casa: {"gols": pred['gols_casa']},
        fora: {"gols": pred['gols_fora']},
    }

with open('previsoes_copa2026.json', 'w', encoding='utf-8') as f:
    json.dump(saida_json, f, ensure_ascii=False, indent=2)

print("✅ JSON gerado: previsoes_copa2026.json")
print()
print(json.dumps(saida_json, ensure_ascii=False, indent=2))


## 12. Resumo do Pipeline

| Etapa | Detalhe |
|---|---|
| **Dados históricos** | 7.515 jogos entre as 48 seleções da Copa (1872–2026) |
| **Features por jogo** | 9 features: 6 diferenças estruturais + 3 probabilidades de odds |
| **Odds no histórico** | Valor neutro 1/3 (sem informação de mercado disponível) |
| **Odds na predição** | Probabilidades reais normalizadas (24 jogos de 2026) |
| **Rótulo** | Placar clippado em 4 gols → 25 classes |
| **Arquitetura** | Input(9) → Dense(128) → Dense(64) → Dense(32) → Softmax(25) |
| **Regularização** | BatchNormalization + Dropout (0.3 / 0.2) |
| **Split** | Treino: até 2021 / Teste: 2022–2026 |
| **Modelo final** | Re-treino em todos os dados históricos |
| **Semente** | `SEED = 42` (Python, NumPy, TensorFlow) |


